# 02 Data Preprocessing

En este notebook se realiza la preparación de los datos para el posterior entrenamiento de los modelos de clasificación.

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

## 1 Carga del dataset e inspección inicial

In [ ]:
CSV_PATH = Path("../data/raw/Telco-Customer-Churn.csv")

df_raw = pd.read_csv(CSV_PATH)

df = df_raw.copy()

In [3]:
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
print(df.dtypes)
df.nunique()

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object


customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

## 2. Preparación y transformación de variables

### 2.1 `TotalCharges`

La variable `TotalCharges` representa los cargos acumulados del cliente, por lo que conceptualmente es una variable numérica.

Durante la inspección se identificaron 11 registros que contienen un espacio en blanco en lugar de un valor numérico. Estos registros corresponden a clientes con `tenure = 0`, es decir, clientes que todavía no han acumulado meses de permanencia.

Por este motivo, `TotalCharges` se convierte a formato numérico y los valores resultantes ausentes se sustituyen por `0`.

In [5]:
df["TotalCharges"].value_counts(dropna=False).head()

TotalCharges
20.2     11
         11
19.75     9
19.9      8
20.05     8
Name: count, dtype: int64

In [6]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [7]:
print("Tipo:", df["TotalCharges"].dtype)
print("Valores nulos:", df["TotalCharges"].isna().sum())

Tipo: float64
Valores nulos: 0


### 2.2 Variable objetivo Churn

`Churn` es la variable objetivo y presenta dos categorías: `Yes` y `No`.

Al tratarse de un problema de clasificación binaria, se transforma a una representación numérica:

- `0`: el cliente permanece (`No`).
- `1`: el cliente abandona (`Yes`).

In [8]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [9]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [10]:
print("Tipo:", df["Churn"].dtype)
print("Valores nulos:", df["Churn"].isna().sum())
print(df["Churn"].value_counts())

Tipo: int64
Valores nulos: 0
Churn
0    5174
1    1869
Name: count, dtype: int64


### 2.3 Identificador: `customerID`

`customerID` es un identificador único para cada cliente y no representa una característica con significado predictivo.

Dado que no se utilizará durante el entrenamiento de los modelos, se elimina del dataset.

In [11]:
df = df.drop(columns=["customerID"])

### 2.4 Variables binarias

Varias variables categóricas presentan únicamente dos posibles estados, por lo que pueden representarse mediante una codificación binaria.

Las variables con valores `Yes` y `No` se transforman utilizando:

- `No` → `0`
- `Yes` → `1`

`SeniorCitizen` ya se encuentra codificada como `0/1`, por lo que no requiere transformación.

La variable `gender` también presenta únicamente dos categorías y se representa mediante una única variable binaria. Esta codificación no implica ningún orden entre sus categorías.

In [12]:
df["gender"] = df["gender"].map({
    "Female": 0,
    "Male": 1
})

In [13]:
binary_yes_no = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

for col in binary_yes_no:
    df[col] = df[col].map({
        "No": 0,
        "Yes": 1
    })

In [14]:
binary_features = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]

for col in binary_features:
    print(f"{col}: {sorted(df[col].unique())}")

gender: [np.int64(0), np.int64(1)]
SeniorCitizen: [np.int64(0), np.int64(1)]
Partner: [np.int64(0), np.int64(1)]
Dependents: [np.int64(0), np.int64(1)]
PhoneService: [np.int64(0), np.int64(1)]
PaperlessBilling: [np.int64(0), np.int64(1)]


### 3.5 Variables asociadas a servicios

Algunas variables presentan categorías adicionales que indican que el cliente no dispone del servicio principal necesario.

En `MultipleLines`, la categoría `No phone service` indica que el cliente no dispone de servicio telefónico. Dado que esta información ya está representada por `PhoneService`, `MultipleLines` se transforma para indicar únicamente si el cliente dispone de múltiples líneas:

- `Yes` → `1`
- `No` → `0`
- `No phone service` → `0`

De forma similar, las variables `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV` y `StreamingMovies` contienen la categoría `No internet service`.

Como la disponibilidad y el tipo de conexión a Internet se encuentran representados por `InternetService`, estas variables se transforman para indicar únicamente si el cliente tiene contratado cada servicio adicional:

- `Yes` → `1`
- `No` → `0`
- `No internet service` → `0`

De esta forma se evita representar como categorías independientes estados cuya información ya está contenida en `PhoneService` e `InternetService`.

In [15]:
df["MultipleLines"] = df["MultipleLines"].map({
    "No phone service": 0,
    "No": 0,
    "Yes": 1
})

In [16]:
print(df["MultipleLines"].value_counts(dropna=False))
print("Valores nulos:", df["MultipleLines"].isna().sum())

MultipleLines
0    4072
1    2971
Name: count, dtype: int64
Valores nulos: 0


In [17]:
internet_services = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for col in internet_services:
    df[col] = df[col].map({
        "No internet service": 0,
        "No": 0,
        "Yes": 1
    })

In [18]:
for col in internet_services:
    print(
        f"{col}: {sorted(df[col].unique())} | "
        f"NaN: {df[col].isna().sum()}"
    )

OnlineSecurity: [np.int64(0), np.int64(1)] | NaN: 0
OnlineBackup: [np.int64(0), np.int64(1)] | NaN: 0
DeviceProtection: [np.int64(0), np.int64(1)] | NaN: 0
TechSupport: [np.int64(0), np.int64(1)] | NaN: 0
StreamingTV: [np.int64(0), np.int64(1)] | NaN: 0
StreamingMovies: [np.int64(0), np.int64(1)] | NaN: 0


### 2.6 Variables categóricas nominales

Tras las transformaciones anteriores, permanecen tres variables categóricas con más de dos categorías:

- `InternetService`
- `Contract`
- `PaymentMethod`

Estas variables se transformarán posteriormente mediante One-Hot Encoding.

El encoder se ajustará únicamente utilizando el conjunto de entrenamiento.
La transformación aprendida se aplicará posteriormente al conjunto de test, evitando que el proceso de ajuste utilice información procedente de dicho conjunto.

In [19]:
categorical_features = [
    "InternetService",
    "Contract",
    "PaymentMethod"
]

for col in categorical_features:
    print(f"{col}: {df[col].unique()}")

InternetService: <StringArray>
['DSL', 'Fiber optic', 'No']
Length: 3, dtype: str
Contract: <StringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str
PaymentMethod: <StringArray>
[         'Electronic check',              'Mailed check',
 'Bank transfer (automatic)',   'Credit card (automatic)']
Length: 4, dtype: str


## 3. Separación de variables predictoras y variable objetivo

Se separa la variable objetivo `Churn` del conjunto de variables predictoras.

- `X` contiene las características utilizadas para realizar las predicciones.
- `y` contiene la variable objetivo que indica si el cliente abandona o no.

In [20]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [21]:
print("X:", X.shape)
print("y:", y.shape)

X.head()

X: (7043, 19)
y: (7043,)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,0,0,1,0,1,0,0,DSL,0,1,0,0,0,0,Month-to-month,1,Electronic check,29.85,29.85
1,1,0,0,0,34,1,0,DSL,1,0,1,0,0,0,One year,0,Mailed check,56.95,1889.50
2,1,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15
3,1,0,0,0,45,0,0,DSL,1,0,1,1,0,0,One year,0,Bank transfer (automatic),42.30,1840.75
4,0,0,0,0,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65


In [22]:
y.value_counts(normalize=True)

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

## 4. División en entrenamiento y test

El dataset se divide en dos subconjuntos:

- **Train (80 %):** utilizado para ajustar el preprocesamiento y entrenar los modelos.
- **Test (20 %):** reservado para evaluar el modelo final sobre observaciones no utilizadas durante el entrenamiento.

Se utiliza `stratify=y` para mantener aproximadamente la misma proporción de clientes con y sin churn en ambos conjuntos.

Se fija `random_state=42` para garantizar que la división sea reproducible.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [24]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (5634, 19)
X_test: (1409, 19)
y_train: (5634,)
y_test: (1409,)


In [25]:
print("Churn dataset completo:")
print(y.value_counts(normalize=True))

print("\nChurn train:")
print(y_train.value_counts(normalize=True))

print("\nChurn test:")
print(y_test.value_counts(normalize=True))

Churn dataset completo:
Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

Churn train:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

Churn test:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


## 4. One-Hot Encoding

Las variables `InternetService`, `Contract` y `PaymentMethod` se codifican mediante One-Hot Encoding.

El encoder se ajusta exclusivamente utilizando `X_train`. Posteriormente el mismo encoder se utiliza para transformar `X_test`.

`handle_unknown="ignore"` permite procesar futuras categorías no observadas durante el entrenamiento sin provocar un error durante la transformación.

In [27]:
categorical_features = [
    "InternetService",
    "Contract",
    "PaymentMethod"
]

In [28]:
X_train[categorical_features].head()

,InternetService,Contract,PaymentMethod
3738,DSL,Month-to-month,Electronic check
3151,Fiber optic,Month-to-month,Mailed check
4860,DSL,Two year,Mailed check
3867,DSL,Two year,Credit card (automatic)
3810,DSL,Month-to-month,Electronic check


In [29]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
    dtype=int
)

In [30]:
X_train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

In [31]:
for feature, categories in zip(
    categorical_features,
    encoder.categories_
):
    print(f"{feature}: {categories}")

InternetService: ['DSL' 'Fiber optic' 'No']
Contract: ['Month-to-month' 'One year' 'Two year']
PaymentMethod: ['Bank transfer (automatic)' 'Credit card (automatic)' 'Electronic check'
 'Mailed check']


In [32]:
encoded_columns = encoder.get_feature_names_out(
    categorical_features
)

encoded_columns

array(['InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'Contract_Month-to-month',
       'Contract_One year', 'Contract_Two year',
       'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype=object)

In [33]:
X_test_encoded = encoder.transform(
    X_test[categorical_features]
)

In [34]:
print(X_train_encoded.shape)
print(X_test_encoded.shape)
print(encoded_columns)

(5634, 10)
(1409, 10)
['InternetService_DSL' 'InternetService_Fiber optic' 'InternetService_No'
 'Contract_Month-to-month' 'Contract_One year' 'Contract_Two year'
 'PaymentMethod_Bank transfer (automatic)'
 'PaymentMethod_Credit card (automatic)' 'PaymentMethod_Electronic check'
 'PaymentMethod_Mailed check']


### 4.1 Reconstrucción de los conjuntos de datos

Los arrays generados por `OneHotEncoder` se convierten en DataFramesconservando los índices originales. Posteriormente se sustituyen las variables categóricas originales por las nuevas variables binarias.

In [35]:
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=encoded_columns,
    index=X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=encoded_columns,
    index=X_test.index
)

In [36]:
X_train = X_train.drop(columns=categorical_features)
X_test = X_test.drop(columns=categorical_features)

In [37]:
X_train = pd.concat(
    [X_train, X_train_encoded],
    axis=1
)

X_test = pd.concat(
    [X_test, X_test_encoded],
    axis=1
)

In [38]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nColumnas no numéricas en train:")
print(X_train.select_dtypes(exclude="number").columns.tolist())

print("\nColumnas no numéricas en test:")
print(X_test.select_dtypes(exclude="number").columns.tolist())

X_train: (5634, 26)
X_test: (1409, 26)

Columnas no numéricas en train:
[]

Columnas no numéricas en test:
[]


In [39]:
print("Mismas columnas:", X_train.columns.equals(X_test.columns))
print("Nulos train:", X_train.isna().sum().sum())
print("Nulos test:", X_test.isna().sum().sum())

Mismas columnas: True
Nulos train: 0
Nulos test: 0


## 5. Resultado del preprocesamiento

Tras completar el preprocesamiento, los datos quedan divididos en conjuntos de entrenamiento y test con todas las variables predictoras representadas numéricamente.

El conjunto de entrenamiento contiene 5.634 observaciones y el conjunto de test 1.409 observaciones, ambos con 26 variables predictoras.

Las transformaciones que dependan del modelo, como el escalado de las variables numéricas, se realizarán posteriormente dentro del proceso de entrenamiento correspondiente.

## 9. Guardado de los datos procesados

Los conjuntos de entrenamiento y test se almacenan por separado para poder utilizarlos posteriormente durante el entrenamiento y evaluación de los modelos sin necesidad de repetir el proceso de preparación de los datos.

Se guardan tanto las variables predictoras (`X_train`, `X_test`) como la variable objetivo (`y_train`, `y_test`).

In [42]:
PROCESSED_DATA_PATH = "../data/processed"

X_train.to_csv(f"{PROCESSED_DATA_PATH}/X_train.csv", index=False)
X_test.to_csv(f"{PROCESSED_DATA_PATH}/X_test.csv", index=False)
y_train.to_csv(f"{PROCESSED_DATA_PATH}/y_train.csv", index=False)
y_test.to_csv(f"{PROCESSED_DATA_PATH}/y_test.csv", index=False)